# Intégration LFM2.5-Audio dans vLLM-Omni

Sert le modèle **LiquidAI/LFM2.5-Audio** (S2S interleaved texte+audio) via **vLLM-Omni** 
pour le prefix caching, le paged KV et le continuous batching — gains visés sur le 
round-trip tool-calling et le multi-sessions.

Le plugin out-of-tree vit dans le repo `rcarvalo/finetuning_s2s_toolcalling` 
(branche `claude/blissful-tesla-7i1yky`), package `lfm2_audio.vllm_plugin`. Ce notebook le clone, l'installe, 
convertit le checkpoint, et valide la chaîne de bout en bout.

> **Runtime requis : GPU** (Exécution → Modifier le type d'exécution → T4/L4/A100). 
> L'engine `vllm-omni` est CUDA-only — il ne tourne pas sur CPU/Mac.

Chaque cellule encode un écart runtime déjà diagnostiqué (vllm-omni 0.22.0) — exécute-les dans l'ordre.

## 1. Vérification du GPU

In [ ]:
import sys
print('Python', sys.version.split()[0])
!nvidia-smi -L || echo 'AUCUN GPU — changer le type d execution avant de continuer'

## 2. Clone du repo (plugin)

Idempotent : re-exécutable sans risque (pull si déjà cloné).

In [ ]:
import os
REPO = 'https://github.com/rcarvalo/finetuning_s2s_toolcalling.git'
BRANCH = 'claude/blissful-tesla-7i1yky'
DST = '/content/finetuning_s2s_toolcalling'
if not os.path.isdir(DST):
    !git clone -q -b {BRANCH} {REPO} {DST}
%cd {DST}
!git pull --ff-only -q && git log --oneline -1

## 3. Installation des dépendances

**Écart 1** — `vllm-omni` ne déclare PAS `vllm` en dépendance : on l'installe séparément 
à la version appariée (`0.22.0`). On utilise `sys.executable -m pip` (le `!pip` de Colab 
peut viser un autre interpréteur que le kernel). Compter ~6-8 min sur VM fraîche.

In [ ]:
# 3. Dépendances.
#    vllm-omni ne déclare pas vllm (versions appariées major.minor) et le build
#    PyPI de vllm 0.22 est CUDA 13 (libcudart.so.13, absent de Colab) →
#    wheel officiel +cu129 du release GitHub + torch assorti (index cu129).
import subprocess, sys
VLLM_WHL = "https://github.com/vllm-project/vllm/releases/download/v0.22.1/vllm-0.22.1+cu129-cp38-abi3-manylinux_2_28_x86_64.whl"
TORCH_IDX = "https://download.pytorch.org/whl/cu129"
!{sys.executable} -m pip install -q "vllm @ {VLLM_WHL}" --extra-index-url {TORCH_IDX}

# Garde-fou : si une tentative précédente a laissé un torch CUDA 13, on le réaligne.
cuda = subprocess.run([sys.executable, "-c", "import torch; print(torch.version.cuda)"],
                      capture_output=True, text=True).stdout.strip()
print("torch CUDA:", cuda or "?")
if cuda.startswith("13"):
    tv = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__.split('+')[0])"],
                        capture_output=True, text=True).stdout.strip()
    !{sys.executable} -m pip install -q --force-reinstall --no-deps "torch=={tv}+cu129" --index-url {TORCH_IDX}
    print("⚠️ torch réaligné sur cu129 — Exécution → Redémarrer la session, puis reprendre ici.")

!{sys.executable} -m pip install -q "vllm-omni==0.22.0" "liquid-audio>=1.3.0"
!{sys.executable} -m pip install -q -e /content/finetuning_s2s_toolcalling --no-deps
import importlib.metadata as md
print("vllm", md.version("vllm"), "| vllm-omni", md.version("vllm-omni"),
      "| liquid-audio", md.version("liquid-audio"))


## 4. Correctif des libs CUDA (écart 2)

Le wheel `vllm` 0.22.0 est buildé pour **CUDA 13** alors que le torch de Colab est en cu12x : 
`import vllm_omni` échoue sur `libcudart.so.13` introuvable. On précharge les libs `cu13` 
(ctypes, pour ce kernel) **et** on exporte `LD_LIBRARY_PATH` (hérité par les sous-process 
`StageEngineCoreProc` que l'engine spawn). À exécuter **avant** tout import de `vllm_omni`.

In [ ]:
import os, glob, ctypes
cu13_dirs = glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cu13/lib')
assert cu13_dirs, 'paquet nvidia cu13 introuvable (réexécuter la cellule 3 ?)'
cu13 = cu13_dirs[0]
os.environ['LD_LIBRARY_PATH'] = cu13 + ':' + os.environ.get('LD_LIBRARY_PATH', '')
for so in sorted(glob.glob(cu13 + '/lib*.so*')):
    try:
        ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
    except OSError:
        pass
import vllm, vllm_omni
print('vllm', vllm.__version__, '| vllm_omni', vllm_omni.__version__, '— imports OK')

## 5. Smoke-test progressif (imports → plugin → contrat runtime)

Vérifie que l'entry point `lfm2_audio` est découvert, que l'architecture + le pipeline 
sont enregistrés, et que le contrat vllm-omni 0.22.0 (hook `sample()`, champs 
`StagePipelineConfig`) est présent.

In [ ]:
import sys
!cd /content/finetuning_s2s_toolcalling && {sys.executable} scripts/colab_smoke_vllm_omni.py

## 6. Conversion du checkpoint

Télécharge la base `LiquidAI/LFM2.5-Audio-1.5B` et la convertit au layout vLLM-Omni 
(`config.json` : `model_type=lfm2_audio`, architecture, ids placeholders, ratio interleaved). 
Pour servir TON modèle finetuné FR : remplace `BASE` par le chemin de ton export `--mode full`.

In [ ]:
import sys, os
from huggingface_hub import snapshot_download
BASE = snapshot_download('LiquidAI/LFM2.5-Audio-1.5B')
OMNI = '/content/lfm25_audio_omni'
if not os.path.exists(os.path.join(OMNI, 'config.json')):
    !cd /content/finetuning_s2s_toolcalling && {sys.executable} -m lfm2_audio.vllm_plugin.convert_checkpoint --checkpoint {BASE} --output {OMNI}
!cd /content/finetuning_s2s_toolcalling && {sys.executable} scripts/colab_smoke_vllm_omni.py --checkpoint {OMNI}

## 7. Démarrage de l'engine + génération texte (E2E)

Lance les 2 stages (AR interleaved → détokeniseur) et génère. Flags issus de l'itération :

- **écart 3** : `load_omni_general_plugins()` AVANT `Omni()` (la détection de pipeline 
  précède le chargement des plugins par l'engine) ;
- **écart 4** : un `SamplingParams` PAR stage (`num_stages`) ;
- **écart 5** : `async_scheduling=False` (le scheduling async tronque l'historique remis 
  au sampler custom → décale la machine à états interleaved d'un step) ;
- `enforce_eager=True`, `gpu_memory_utilization=0.42` (2 stages sur 1 GPU), `dtype=float16` 
  (T4 sans bf16 ; mettre `bfloat16` sur A100/L4 pour la parité), `stage_init_timeout=1200` (T4 lent).

Démarrage ~4-6 min (chargement fp16 des 2 stages + profiling).

In [ ]:
import sys, os
SRC = '/content/finetuning_s2s_toolcalling/src'
sys.path.insert(0, SRC)                                          # ce kernel
os.environ['PYTHONPATH'] = SRC + ':' + os.environ.get('PYTHONPATH', '')  # workers spawn

import vllm_omni.plugins as _p
_p.omni_plugins_loaded = False        # un 1er essai a échoué → forcer le rechargement
import lfm2_audio.vllm_plugin           # doit importer sans erreur maintenant

from vllm_omni.plugins import load_omni_general_plugins
load_omni_general_plugins()


In [ ]:
import gc, torch
for n in ('model', 'proc', 'omni'):
    if n in globals():
        del globals()[n]
gc.collect(); torch.cuda.empty_cache()
print('VRAM libre:', torch.cuda.mem_get_info()[0] / 1e9, 'Go')


In [ ]:
import sys, os
SRC = '/content/finetuning_s2s_toolcalling/src'
sys.path.insert(0, SRC)                                          # ce kernel
os.environ['PYTHONPATH'] = SRC + ':' + os.environ.get('PYTHONPATH', '')  # workers spawn
import vllm_omni.plugins as _p; _p.omni_plugins_loaded = False   # un essai a pu échouer
import lfm2_audio.vllm_plugin                                      # doit importer
from vllm_omni.plugins import load_omni_general_plugins; load_omni_general_plugins()

from vllm import SamplingParams
from vllm_omni import Omni
from transformers import AutoTokenizer
OMNI = '/content/lfm25_audio_omni'
tok = AutoTokenizer.from_pretrained(OMNI)

def build_prompt_ids(user_text):
    text = ('<|startoftext|><|im_start|>system\n'
            'Respond with interleaved text and audio.<|im_end|>\n'
            f'<|im_start|>user\n{user_text}<|im_end|>\n'
            '<|im_start|>assistant\n')
    return tok(text, add_special_tokens=False).input_ids

# deploy_config = LE point clé : sans lui, ni connector streaming ni prefix caching
# (il porte async_chunk, dtype bfloat16, SharedMemoryConnector codec_streaming,
#  enable_prefix_caching, split mémoire par stage).
omni = Omni(
    model=OMNI,
    deploy_config='/content/finetuning_s2s_toolcalling/configs/serving/vllm_omni.yaml',
    stage_init_timeout=1200,
    init_timeout=1800,
print('engine prêt — async_chunk:', omni.async_chunk)

## 8. Génération interleaved : texte + frames audio

Greedy (parité), 2 `SamplingParams`. On inspecte les ids du stage 0 (placeholders frame/EOA = 
audio généré), la cadence, et on récupère le waveform du stage 1.

In [ ]:
from lfm2_audio.vllm_plugin.constants import (
    AUDIO_FRAME_PLACEHOLDER_ID, AUDIO_EOA_PLACEHOLDER_ID, IM_END_TOKEN_ID)

prompt_ids = build_prompt_ids('Bonjour, qui es-tu ?')
sp0 = SamplingParams(temperature=0.0, max_tokens=128, stop_token_ids=[IM_END_TOKEN_ID])
sp1 = SamplingParams(max_tokens=1, detokenize=False)
outputs = omni.generate({'prompt_token_ids': prompt_ids}, [sp0, sp1])

wav = None
for out in outputs:
    ro = out.request_output
    if out.final_output_type == 'text' and ro is not None and ro.outputs:
        ids = list(ro.outputs[0].token_ids)
        n_frames = sum(i == AUDIO_FRAME_PLACEHOLDER_ID for i in ids)
        n_eoa = sum(i == AUDIO_EOA_PLACEHOLDER_ID for i in ids)
        text_ids = [i for i in ids if i not in (AUDIO_FRAME_PLACEHOLDER_ID, AUDIO_EOA_PLACEHOLDER_ID)]
        print(f'stage 0 : {len(ids)} ids — {len(text_ids)} texte, {n_frames} frames, {n_eoa} EOA')
        print('texte :', repr(ro.outputs[0].text[:200]))
    elif out.final_output_type == 'audio':
        mm = getattr(ro, 'multimodal_output', None) or getattr(out, 'multimodal_output', None)
        print('stage 1 : sortie audio —', type(mm).__name__)
        wav = mm
print('frames audio générées' if n_frames else '[warn] aucune frame audio')

## 9. Écoute du résultat

Reconstruit un tensor 1D depuis la sortie du stage 1 (12,5 frames/s → 24 kHz) et le joue. 
Si la structure de `multimodal_output` diffère, la cellule l'introspecte pour ajuster.

In [ ]:
import numpy as np, torch
from IPython.display import Audio, display

def to_waveform(x):
    if x is None:
        return None
    if isinstance(x, dict):
        for k in ('model_outputs', 'audio', 'waveform', 'wav'):
            if k in x:
                return to_waveform(x[k])
    if isinstance(x, torch.Tensor):
        return x.detach().float().cpu().numpy().reshape(-1)
    if isinstance(x, np.ndarray):
        return x.reshape(-1)
    return None

w = to_waveform(wav)
if w is not None and w.size:
    print('waveform :', w.shape, f'({w.size/24000:.2f}s @ 24kHz)')
    display(Audio(w, rate=24000))
else:
    print('pas de waveform exploitable — structure brute :')
    print(repr(wav)[:500])

## 10. Sonde automatique (optionnel)

Relance la chaîne via le script du repo (mêmes flags) avec un diagnostic compact — pratique 
pour itérer après un `git pull`. Redémarre l'engine dans un sous-process propre.

In [ ]:
import sys, os
env = dict(os.environ)
!cd /content/finetuning_s2s_toolcalling && LD_LIBRARY_PATH={env['LD_LIBRARY_PATH']} {sys.executable} scripts/colab_probe_interleaved.py --checkpoint /content/lfm25_audio_omni 2>&1 | tail -20

## Où on en est

| Étape | État |
|---|---|
| Plugin chargé, pipeline 2-stages enregistré | ✅ |
| Engine démarre, génère texte E2E | ✅ |
| Stage 0 : machine à états interleaved (sample/depthformer) | ✅ active en runtime |
| **Parité greedy vs `liquid_audio.generate_interleaved`** | ⏳ bloquant P2 |
| Stage 1 : parité waveform + TTFA | ⏳ P3 |

Pour servir le **modèle finetuné FR + tool calling** : exporter avec 
`training/export_checkpoint.py --mode full`, pointer `BASE` (cellule 6) sur cet export, 
le ratio interleaved calibré est lu depuis son `config.json`.

In [ ]:
import time, numpy as np, torch
from IPython.display import Audio, display
from vllm import SamplingParams
from lfm2_audio.vllm_plugin.constants import (
    AUDIO_FRAME_PLACEHOLDER_ID, AUDIO_EOA_PLACEHOLDER_ID, IM_END_TOKEN_ID)

SYS = "You are a helpful assistant that can respond with interleaved text and audio. Use the following format:\n" 
history = []  # (role, texte) — on ne garde que le texte en contexte (pas les placeholders)

def _render(h):
    s = "<|startoftext|><|im_start|>system\n" + SYS + "<|im_end|>\n"
    for role, txt in h:
        s += f"<|im_start|>{role}\n{txt}<|im_end|>\n"
    return tok(s + "<|im_start|>assistant\n", add_special_tokens=False).input_ids

def _wave(x):
    if isinstance(x, dict):
        for k in ('model_outputs', 'audio', 'waveform', 'wav'):
            if k in x: return _wave(x[k])
    if isinstance(x, torch.Tensor): return x.detach().float().cpu().numpy().reshape(-1)
    if isinstance(x, np.ndarray): return x.reshape(-1)
    return None

def say(user_text, max_tokens=256):
    history.append(("user", user_text))
    ids = _render(history)
    sp0 = SamplingParams(temperature=0.0, max_tokens=max_tokens, stop_token_ids=[IM_END_TOKEN_ID])
    sp1 = SamplingParams(max_tokens=1, detokenize=False)
    t0 = time.time()
    outs = omni.generate({"prompt_token_ids": ids}, [sp0, sp1])
    dt = time.time() - t0
    text, wav, nf = "", None, 0
    for o in outs:
        ro = o.request_output
        if o.final_output_type == "text" and ro and ro.outputs:
            toks = list(ro.outputs[0].token_ids)
            nf = sum(t in (AUDIO_FRAME_PLACEHOLDER_ID, AUDIO_EOA_PLACEHOLDER_ID) for t in toks)
            text = ro.outputs[0].text
        elif o.final_output_type == "audio":
            wav = _wave(getattr(ro, "multimodal_output", None) or getattr(o, "multimodal_output", None))
    history.append(("assistant", text))
    print(f"🤖 {text}\n   {nf} frames · {nf/12.5:.1f}s audio · généré en {dt:.1f}s")
    if wav is not None and wav.size:
        display(Audio(wav, rate=24000))
    return text

print("Dialogue — message vide pour quitter.")
while True:
    u = input("👤 ").strip()
    if not u: break
    say(u)


In [ ]:
# === Référence liquid-audio (vérité terrain, SANS vLLM) ===
# Runtime frais + GPU. C'est la sortie "correcte" attendue du modèle de base.
import torch
from IPython.display import Audio, display
from liquid_audio import LFM2AudioModel, LFM2AudioProcessor, ChatState

model = LFM2AudioModel.from_pretrained("LiquidAI/LFM2.5-Audio-1.5B", device="cuda").eval()
proc  = LFM2AudioProcessor.from_pretrained("LiquidAI/LFM2.5-Audio-1.5B", device="cuda")

def ref_say(user_text, max_new_tokens=256):
    chat = ChatState(proc)
    chat.new_turn("system"); chat.add_text("Respond with interleaved text and audio."); chat.end_turn()
    chat.new_turn("user");   chat.add_text(user_text); chat.end_turn()
    chat.new_turn("assistant")
    text_ids, frames = [], []
    with torch.no_grad():
        for t in model.generate_interleaved(**chat, max_new_tokens=max_new_tokens):
            (text_ids if t.numel() == 1 else frames).append(t.detach().cpu())
    txt = proc.text.decode([int(x) for x in text_ids])
    print("🤖", txt, "|", len(frames), "frames")
    keep = [f.flatten() for f in frames if int(f.flatten()[0]) != 2048]
    if keep:
        fr = torch.stack(keep, dim=1).cuda()          # (8, T)
        wav = proc.decode(fr.unsqueeze(0))            # (1, 8, T) -> waveform
        display(Audio(wav.float().cpu().numpy().reshape(-1), rate=24000))

ref_say("Hello, who are you?")


In [ ]:

def ref_say(user_text, max_new_tokens=256):
    chat = ChatState(proc)
    chat.new_turn("system"); chat.add_text("Respond with interleaved text and audio."); chat.end_turn()
    chat.new_turn("user");   chat.add_text(user_text); chat.end_turn()
    chat.new_turn("assistant")
    text_ids, frames = [], []
    with torch.no_grad():
        for t in model.generate_interleaved(**chat, max_new_tokens=max_new_tokens):
            (text_ids if t.numel() == 1 else frames).append(t.detach().cpu())
    txt = proc.text.decode([int(x) for x in text_ids])
    print("🤖", txt, "|", len(frames), "frames")
    keep = [f.flatten() for f in frames if int(f.flatten()[0]) != 2048]
    if keep:
        fr = torch.stack(keep, dim=1).cuda()          # (8, T)
        wav = proc.decode(fr.unsqueeze(0))            # (1, 8, T) -> waveform
        display(Audio(wav.float().cpu().numpy().reshape(-1), rate=24000))

ref_say("Hello, who are you?")


In [ ]:
import time
from IPython.display import Audio, display
#⚠️ nécessite un engine relancé avec async_chunk=True :
#   omni = Omni(model=OMNI, dtype='bfloat16', enforce_eager=True,
#               gpu_memory_utilization=0.42, async_scheduling=False,
#               async_chunk=True, stage_init_timeout=1200)

prompt_ids = build_prompt_ids("Bonjour, qui es-tu ?")
sp0 = SamplingParams(temperature=0.0, max_tokens=256, stop_token_ids=[IM_END_TOKEN_ID])
sp1 = SamplingParams(max_tokens=1, detokenize=False)

t0 = time.time(); ttfa = None; chunks = []
for out in omni.generate({'prompt_token_ids': prompt_ids}, [sp0, sp1], py_generator=True):
    if out.final_output_type == 'audio':
        w = to_waveform(getattr(out.request_output, 'multimodal_output', None))
        if w is not None and w.size:
            if ttfa is None:
                ttfa = time.time() - t0
                print(f"⏱️ TTFA = {ttfa*1000:.0f} ms")
            chunks.append(w)
print(f"{len(chunks)} chunks · total {time.time()-t0:.1f}s")
if chunks:
    import numpy as np
    display(Audio(np.concatenate(chunks), rate=24000))
